# 🏁 Allan QLoRA — Шаг 3: Объединение адаптеров и экспорт

Этот ноутбук:
1. Показывает все сохранённые LoRA-адаптеры
2. Объединяет выбранный адаптер с базовой моделью (merge)
3. Экспортирует в формат GGUF для использования в Ollama/LM Studio
4. Опционально: загружает на HuggingFace Hub

---
**Требуется:** выполненные ноутбуки 01 и минимум один цикл 02.

## Шаг 0: Установка и монтирование

In [ ]:
!pip install -q transformers==4.44.2 peft==0.12.0 accelerate==0.33.0 \
    bitsandbytes==0.43.3 sentencepiece protobuf huggingface_hub

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Drive подключён")

## Шаг 1: Просмотр всех адаптеров

In [ ]:
import json
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')

# Загрузка конфига и прогресса
with open(DRIVE_ROOT / 'config.json') as f:
    project_config = json.load(f)
with open(DRIVE_ROOT / 'progress.json') as f:
    progress = json.load(f)

BASE_MODEL = project_config.get('base_model', 'Qwen/Qwen2.5-7B-Instruct')

print("=" * 60)
print("ДОСТУПНЫЕ АДАПТЕРЫ")
print("=" * 60)
print(f"Базовая модель: {BASE_MODEL}\n")

adapters_dir = DRIVE_ROOT / 'adapters'
all_adapters = []

for ds_name, info in progress.items():
    adapter_paths = info.get('adapter_paths', {})
    if adapter_paths:
        print(f"📦 {ds_name} ({info['description']}):")
        for chunk_idx, path in sorted(adapter_paths.items(), key=lambda x: int(x[0])):
            p = Path(path)
            exists = p.exists()
            size_str = ''
            if exists:
                import subprocess
                r = subprocess.run(['du', '-sh', str(p)], capture_output=True, text=True)
                size_str = r.stdout.split()[0] if r.stdout else '?'
            status = f"✅ ({size_str})" if exists else "❌ НЕ НАЙДЕН"
            print(f"   Чанк {chunk_idx}: {p.name} {status}")
            if exists:
                all_adapters.append((ds_name, int(chunk_idx), str(p)))

print(f"\n📊 Итого адаптеров: {len(all_adapters)}")

if not all_adapters:
    print("\n❌ Адаптеры не найдены. Сначала выполните ноутбук 02.")
else:
    print("\n✅ Адаптеры найдены. Можно объединять и экспортировать.")

## Шаг 2: Выбор адаптера для объединения

Можно объединить:
- **Последний** адаптер конкретного датасета (рекомендуется)
- **Любой** конкретный адаптер по пути

In [ ]:
# ================================================================
# НАСТРОЙКА: Выберите адаптер для объединения
# ================================================================

# Режим выбора:
# 'latest'  — последний адаптер из указанного датасета
# 'path'    — конкретный путь к адаптеру
ADAPTER_SELECT_MODE = 'latest'

# При ADAPTER_SELECT_MODE = 'latest': выберите датасет
# 'AUTO' — берёт датасет с наибольшим количеством обученных чанков
ADAPTER_DATASET = 'AUTO'

# При ADAPTER_SELECT_MODE = 'path': укажите путь вручную
ADAPTER_PATH_MANUAL = '/content/drive/MyDrive/AllanQLoRA/adapters/ru_turbo_alpaca_chunk0000'

# Имя финальной модели
OUTPUT_MODEL_NAME = 'allan_ru_qlora'

# ================================================================

import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
with open(DRIVE_ROOT / 'progress.json') as f:
    progress = json.load(f)

selected_adapter_path = None

if ADAPTER_SELECT_MODE == 'latest':
    # Находим датасет с наибольшим прогрессом
    if ADAPTER_DATASET == 'AUTO':
        best_ds = max(
            [(ds, len(info['completed_chunks'])) for ds, info in progress.items() if info.get('adapter_paths')],
            key=lambda x: x[1],
            default=(None, 0)
        )[0]
    else:
        best_ds = ADAPTER_DATASET
    
    if best_ds and progress[best_ds].get('adapter_paths'):
        adapter_paths = progress[best_ds]['adapter_paths']
        # Берём адаптер с максимальным индексом чанка
        last_chunk_idx = max(adapter_paths.keys(), key=lambda x: int(x))
        selected_adapter_path = adapter_paths[last_chunk_idx]
        print(f"✅ Выбран адаптер: {best_ds}, чанк {last_chunk_idx}")
    else:
        raise ValueError("❌ Адаптеры не найдены. Запустите ноутбук 02.")

elif ADAPTER_SELECT_MODE == 'path':
    selected_adapter_path = ADAPTER_PATH_MANUAL
    print(f"✅ Выбран адаптер по пути: {selected_adapter_path}")

print(f"\n📂 Путь к адаптеру: {selected_adapter_path}")
print(f"📤 Имя выходной модели: {OUTPUT_MODEL_NAME}")

## Шаг 3: Загрузка и объединение (Merge)

Merge объединяет веса LoRA-адаптера с базовой моделью в одну полноценную модель.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
merged_dir = DRIVE_ROOT / 'exports' / f'{OUTPUT_MODEL_NAME}_merged'
merged_dir.mkdir(parents=True, exist_ok=True)

print(f"📥 Загрузка базовой модели {BASE_MODEL} (fp16)...")
print("   (без квантизации для merge)\n")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print(f"✅ Базовая модель загружена")
print(f"   VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} ГБ")

print(f"\n🔧 Загрузка LoRA-адаптера из {selected_adapter_path}...")
peft_model = PeftModel.from_pretrained(
    base_model,
    selected_adapter_path,
)

print("\n🔧 Объединение весов (merge_and_unload)...")
merged_model = peft_model.merge_and_unload()

print(f"\n💾 Сохранение объединённой модели в {merged_dir}...")
merged_model.save_pretrained(str(merged_dir), safe_serialization=True)
tokenizer.save_pretrained(str(merged_dir))

import subprocess
r = subprocess.run(['du', '-sh', str(merged_dir)], capture_output=True, text=True)
print(f"✅ Сохранено! Размер: {r.stdout.split()[0]}")
print(f"   Путь: {merged_dir}")

## Шаг 4: Экспорт в GGUF (для Ollama / LM Studio)

GGUF позволяет запускать модель на CPU/GPU без Python.

In [ ]:
# ================================================================
# НАСТРОЙКА: Квантизация GGUF
# q4_k_m  — 4-bit (хорошее качество, ~4 ГБ для 7B)  [РЕКОМЕНДУЕТСЯ]
# q5_k_m  — 5-bit (лучше качество, ~5 ГБ)
# q8_0    — 8-bit (почти без потерь, ~8 ГБ)
# f16     — fp16 (без квантизации, ~14 ГБ) — только для reference
# ================================================================
GGUF_QUANT = 'q4_k_m'
# ================================================================

import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
merged_dir = DRIVE_ROOT / 'exports' / f'{OUTPUT_MODEL_NAME}_merged'
gguf_dir = DRIVE_ROOT / 'exports' / 'gguf'
gguf_dir.mkdir(parents=True, exist_ok=True)

gguf_output = gguf_dir / f'{OUTPUT_MODEL_NAME}_{GGUF_QUANT}.gguf'

# Клонируем llama.cpp для конвертации
print("📥 Клонирование llama.cpp для конвертации в GGUF...")
!git clone --depth 1 -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
print("✅ llama.cpp готов")

# Шаг 4.1: Конвертация в GGUF f16
gguf_f16 = gguf_dir / f'{OUTPUT_MODEL_NAME}_f16.gguf'
print(f"\n🔧 Конвертация в GGUF (f16)...")

convert_result = subprocess.run(
    [
        'python3', '/content/llama.cpp/convert_hf_to_gguf.py',
        str(merged_dir),
        '--outfile', str(gguf_f16),
        '--outtype', 'f16',
    ],
    capture_output=True, text=True
)

if convert_result.returncode != 0:
    print("❌ Ошибка конвертации:")
    print(convert_result.stderr[-2000:])
else:
    print(f"✅ f16 GGUF создан: {gguf_f16}")
    
    # Шаг 4.2: Квантизация
    if GGUF_QUANT != 'f16':
        print(f"\n🔧 Квантизация {GGUF_QUANT}...")
        
        # Компилируем quantize если нет
        if not Path('/content/llama.cpp/llama-quantize').exists():
            !cd /content/llama.cpp && cmake -B build -DLLAMA_CUDA=ON 2>/dev/null || \
             cmake -B build
            !cd /content/llama.cpp && cmake --build build --config Release -j$(nproc) -t llama-quantize
            # Ищем бинарник
            !find /content/llama.cpp/build -name 'llama-quantize' -exec cp {} /content/llama.cpp/ \; 2>/dev/null || true
        
        quant_bin = '/content/llama.cpp/build/bin/llama-quantize'
        if not Path(quant_bin).exists():
            quant_bin = '/content/llama.cpp/llama-quantize'
        
        quant_result = subprocess.run(
            [quant_bin, str(gguf_f16), str(gguf_output), GGUF_QUANT.upper()],
            capture_output=True, text=True
        )
        
        if quant_result.returncode != 0:
            print("⚠️  Ошибка квантизации (возможно нужно перекомпилировать llama.cpp):")
            print(quant_result.stderr[-1000:])
            print(f"\n💡 Используйте f16 GGUF: {gguf_f16}")
            gguf_output = gguf_f16
        else:
            # Удаляем промежуточный f16
            gguf_f16.unlink(missing_ok=True)
            r = subprocess.run(['du', '-sh', str(gguf_output)], capture_output=True, text=True)
            print(f"✅ GGUF {GGUF_QUANT} создан: {gguf_output}")
            print(f"   Размер: {r.stdout.split()[0]}")
    else:
        gguf_output = gguf_f16

## Шаг 5: Создание Modelfile для Ollama

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
gguf_dir = DRIVE_ROOT / 'exports' / 'gguf'

modelfile_content = f'''# Ollama Modelfile для {OUTPUT_MODEL_NAME}
# Сгенерировано автоматически Allan QLoRA pipeline

FROM ./{OUTPUT_MODEL_NAME}_{GGUF_QUANT}.gguf

TEMPLATE """{{{{ if .System }}}}Система: {{{{ .System }}}}\n\n{{{{ end }}}}{{{{ if .Prompt }}}}Пользователь: {{{{ .Prompt }}}}\n\nАссистент: {{{{ end }}}}{{{{ .Response }}}}"""

SYSTEM "Ты — полезный русскоязычный ИИ-ассистент по имени Аллан. Отвечай на русском языке, будь точным и информативным."

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER stop "Пользователь:"
PARAMETER stop "### Задание:"
'''

modelfile_path = gguf_dir / 'Modelfile'
modelfile_path.write_text(modelfile_content, encoding='utf-8')

print(f"✅ Modelfile создан: {modelfile_path}")
print("\nДля использования в Ollama:")
print(f"  ollama create {OUTPUT_MODEL_NAME} -f ./Modelfile")
print(f"  ollama run {OUTPUT_MODEL_NAME}")
print("\n--- Содержимое Modelfile ---")
print(modelfile_content)

## Шаг 6: Тест объединённой модели

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
merged_dir = DRIVE_ROOT / 'exports' / f'{OUTPUT_MODEL_NAME}_merged'

print(f"📥 Загрузка объединённой модели для теста...")

test_tokenizer = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)
test_model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
test_model.eval()
print("✅ Модель загружена")

# Тестовые запросы на русском
test_prompts = [
    "### Задание:\nЧто такое нейронная сеть? Объясни кратко.\n\n### Ответ:\n",
    "### Задание:\nПродолжи фразу: В Москве живут...\n\n### Ответ:\n",
    "Пользователь: Привет! Как тебя зовут?\nАссистент:",
]

print("\n=" * 60)
print("ТЕСТ ФИНАЛЬНОЙ МОДЕЛИ")
print("=" * 60)

for i, prompt in enumerate(test_prompts):
    print(f"\n[Тест {i+1}] {prompt.split(chr(10))[0][:60]}")
    
    inputs = test_tokenizer(prompt, return_tensors='pt').to(test_model.device)
    
    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=test_tokenizer.eos_token_id,
        )
    
    response = test_tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    print(f"Ответ: {response[:400]}")

## Шаг 7: Публикация на HuggingFace Hub (опционально)

In [ ]:
# ================================================================
# НАСТРОЙКА: Загрузка на HuggingFace Hub
# ================================================================
UPLOAD_TO_HF = False          # True — загрузить, False — пропустить
HF_USERNAME = 'your_username' # ваш никнейм на HuggingFace
HF_REPO_NAME = f'{OUTPUT_MODEL_NAME}'
HF_TOKEN = ''                 # токен с HuggingFace (Settings → Access Tokens)
UPLOAD_ADAPTER_ONLY = True    # True — только адаптер (~100МБ), False — полная модель (~14ГБ)
# ================================================================

if UPLOAD_TO_HF:
    if not HF_TOKEN:
        print("❌ Укажите HF_TOKEN для загрузки на HuggingFace")
    else:
        from huggingface_hub import HfApi, login
        from pathlib import Path
        
        login(token=HF_TOKEN)
        api = HfApi()
        
        repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
        
        # Создаём репозиторий
        api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
        print(f"✅ Репозиторий: https://huggingface.co/{repo_id}")
        
        DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
        
        if UPLOAD_ADAPTER_ONLY:
            # Загружаем только LoRA-адаптер (компактно)
            upload_dir = Path(selected_adapter_path)
            print(f"\n📤 Загрузка LoRA-адаптера из {upload_dir}...")
        else:
            # Загружаем полную объединённую модель
            upload_dir = DRIVE_ROOT / 'exports' / f'{OUTPUT_MODEL_NAME}_merged'
            print(f"\n📤 Загрузка полной модели из {upload_dir}...")
        
        api.upload_folder(
            folder_path=str(upload_dir),
            repo_id=repo_id,
            repo_type='model',
        )
        print(f"\n✅ Загружено на: https://huggingface.co/{repo_id}")
else:
    print("⏭️  Загрузка на HuggingFace пропущена (UPLOAD_TO_HF = False)")

## Итоговый отчёт

In [ ]:
import json
import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')

with open(DRIVE_ROOT / 'progress.json') as f:
    progress = json.load(f)

print("=" * 60)
print("ФИНАЛЬНЫЙ ОТЧЁТ")
print("=" * 60)

exports_dir = DRIVE_ROOT / 'exports'
print("\n📦 Экспорты:")
for item in sorted(exports_dir.rglob('*')):
    if item.is_file() and item.suffix in ['.gguf', '.safetensors', '.json', '.bin']:
        r = subprocess.run(['du', '-sh', str(item)], capture_output=True, text=True)
        size = r.stdout.split()[0] if r.stdout else '?'
        print(f"  {size:>8}  {item.name}")

print("\n📊 Прогресс обучения:")
total_done = sum(len(info['completed_chunks']) for info in progress.values())
total_all = sum(info['total_chunks'] for info in progress.values())
total_sessions = sum(len(info['training_sessions']) for info in progress.values())

print(f"  Сессий обучения: {total_sessions}")
print(f"  Чанков обучено:  {total_done}/{total_all}")

print("\n📂 Структура на Drive:")
for subdir in ['adapters', 'exports', 'checkpoints', 'logs']:
    p = DRIVE_ROOT / subdir
    r = subprocess.run(['du', '-sh', str(p)], capture_output=True, text=True)
    size = r.stdout.split()[0] if r.stdout else '?'
    print(f"  {size:>8}  {subdir}/")

print("\n✅ Готово!")